# 07 - Official VGGT-X and MCMC-3DGS baseline

This notebook is an isolated baseline. It does not reuse our custom Gaussian trainer. It runs the released VGGT-X global-alignment export, then the CityGaussian MCMC-3DGS configuration recommended by the authors.

VGGT-X was designed for dense image collections. Using the same eight images is intentional here: it makes the comparison fair, but it is a sparse-input stress test.

## Important environment note

VGGT-X pins Python 3.10-era Torch 2.3.1 and PyCOLMAP 3.10.0. Do not install it into the notebook 05/06 runtime. Use a fresh runtime and follow the official environment installation below. If Colab cannot build the CUDA extensions, run this notebook on a Linux CUDA machine.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)
CODE_ROOT = Path("/content/Project_Thesis_code")
REPOSITORY = "https://github.com/katlit/Project_Thesis.git"
BRANCH = "codex/hq200-example-notebook"
if CODE_ROOT.exists() and not (CODE_ROOT / ".git").is_dir(): shutil.rmtree(CODE_ROOT)
command = (["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, str(CODE_ROOT)]
           if not CODE_ROOT.exists() else ["git", "-C", str(CODE_ROOT), "pull", "--ff-only", "origin", BRANCH])
subprocess.run(command, check=True)
sys.path.insert(0, str(CODE_ROOT / "code"))
PROJECT_ROOT = Path("/content/drive/MyDrive/ITU/3D/Thesis")

import pandas as pd
from PIL import Image
DATASET="3DRealCar"; SCENE=None
manifest=pd.read_csv(PROJECT_ROOT/"data_processed/method_inputs/manifest.csv")
rows=manifest.query("method == 'vggt' and split == 'train' and dataset == @DATASET")
SCENE=SCENE or sorted(rows.scene.unique())[0]; rows=rows[rows.scene.eq(SCENE)].sort_values("view_order")
STAGE=Path("/content/vggtx_input")/SCENE; (STAGE/"images").mkdir(parents=True,exist_ok=True)
for i,row in enumerate(rows.itertuples()):
    Image.open(row.method_image).convert("RGB").save(STAGE/"images"/f"{i:02d}.png")
print("Staged",len(rows),"images at",STAGE)

In [ ]:
# Run these commands in the dedicated Python 3.10 environment described by VGGT-X.
!test -d /content/VGGT-X/.git || git clone --recursive https://github.com/Linketic/VGGT-X.git /content/VGGT-X
print("Official geometry command:")
print(f"python /content/VGGT-X/demo_colmap.py --scene_dir {STAGE} --shared_camera --use_ga --total_frame_num 8")

## Run the official commands

The first command must create a valid COLMAP model and `matches.pt`. Inspect the camera orbit before training. If global alignment destroys the orbit, report the failure rather than silently using it.

After that, follow the official CityGaussian `configs/colmap_pose_opt_mcmc.yaml` command. Save its rendered orbit and metrics under:

`experiments/3DGS/VGGT_X_MCMC/3DRealCar/<scene>/`

This notebook deliberately does not imitate VGGT-X with our trainer. A valid baseline must come from the released VGGT-X and CityGaussian implementations.

In [ ]:
OUTPUT_ROOT=PROJECT_ROOT/"experiments/3DGS/VGGT_X_MCMC"/DATASET/SCENE
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
print("Expected final video:",OUTPUT_ROOT/"closed_orbit.mp4")
print("Expected synchronized frames:",OUTPUT_ROOT/"orbit_frames/view_000.png ...")
print("Expected metrics:",OUTPUT_ROOT/"metrics.csv")